# Free-Droid (Szabi) fine-tuning — 7B/8B size-ceiling test (Unsloth QLoRA, Colab T4)

Same thin-runner pattern as `colab_finetune.ipynb`, but for the **larger cloud-only candidates** (`qwen7b`, `llama8b`). The 3B models produced shaky Hungarian even untuned — this notebook tests whether a 7-8B base lifts the fluency ceiling. All logic lives in `finetune.py` + `config.py`.

**These are CLOUD-ONLY / hybrid candidates** — a Pi 5 can't run a 7-8B in real time. If a larger model wins, the edge keeps a 3B fallback.

**First: Runtime → Change runtime type → T4 GPU.** 7-8B QLoRA is tight on a 15 GB T4, so the variants use `batch_size=1` + `grad_accum=8` (effective batch 8).

In [ ]:
# 1. Install Unsloth (official Colab installer — resolves matching torch/bnb/triton).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Get the repo (dataset + finetune.py + the 7B/8B variants) and switch into training/.
#    The qwen7b/llama8b variants must be on the default branch — merge the PR first.
#    Testing BEFORE merge? clone the branch instead:
#      !git clone --depth 1 --branch larger-model-variants https://github.com/pits2022/free-droid.git
!git clone --depth 1 https://github.com/pits2022/free-droid.git
%cd free-droid/training

In [ ]:
# 3a. Qwen 2.5 7B — gentle recipe. Fits the T4 most reliably; run this one first.
!python finetune.py --variant qwen7b --preset gentle

In [ ]:
# 3b. Llama 3.1 8B — gentle recipe. Tighter on the T4; if it OOMs, add --max-seq-length 1024.
!python finetune.py --variant llama8b --preset gentle

## Next

- Outputs: `training/outputs/<name>-gentle/` — `gguf-q4_k_m` (load this in Ollama), `gguf-q8_0` (cloud), `lora-adapter`. The 8B `q8_0` export is large/slow; `q4_k_m` is enough for the benchmark.
- Download the GGUFs (git-ignored). The export GGUF does **not** carry the chat template, so build the Ollama Modelfile with an **explicit `TEMPLATE`** (Qwen→ChatML, Llama→Llama-3 header) + the Szabi `SYSTEM` — exactly like the 3B `tests/` Modelfiles. Do **not** rely on autodetect (`{{ .Prompt }}` → word-salad + never stops).
- `ollama create freedroid-qwen7b -f Modelfile` / `freedroid-llama8b`, then benchmark against the best 3B:
  ```
  python run_benchmark.py --models freedroid-qwen7b freedroid-llama8b freedroid-llama-gentle
  ```
- Score with `ertekelo_sablon.md`. If the 7-8B Hungarian is clearly better → the magyar + hybrid path is viable; if not, it points toward English (option C) or heavier RAG/dataset work.